In [1]:
import os
import pandas as pd
import wbgapi as wb # https://github.com/tgherzog/wbgapi
import plotly.express as px
import plotly.graph_objects as go
import datetime
import numpy as np
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
# Define your own folder 
out_dir = os.path.join(os.path.expanduser('~'), 'Documents/Websites/JavierParada.github.io/Inventory/')
print(out_dir)

/Users/javierparada/Documents/Websites/JavierParada.github.io/Inventory/


## Check for recent updates in each of the 65 databases

In [ ]:
# Check date for last update of WDI 2021-05-25
ListOfDict = list(wb.source.list())
Databases = pd.DataFrame(ListOfDict)
print("There are {} databases available.".format(len(Databases)))
print("{} was last updated on {}.".format(ListOfDict[1]['name'],ListOfDict[1]['lastupdated']))
Databases['lastupdated'] = pd.to_datetime(Databases['lastupdated'])
Databases.sort_values(by='lastupdated', ascending=False).head()

## 21 topics, 1,443 indicators in WDI, 135 in topic 6 Environment

In [ ]:
# 21 topics
pd.DataFrame(list(wb.topic.list())).head()

In [ ]:
others = ['EG.ELC.ACCS.RU.ZS', 'EG.ELC.ACCS.UR.ZS']
desc_others = pd.DataFrame(list(wb.series.list(others)))
desc = pd.DataFrame(list(wb.series.list(db=2, topic=6)))
desc = desc.append(desc_others)
desc = desc.rename({'id': 'variable', 'value':'description'}, axis=1)
desc.head()

In [ ]:
list(wb.series.list(db=2, topic=6))

# Environment 135

In [ ]:
# List of indicators
others = ['EG.ELC.ACCS.RU.ZS', 'EG.ELC.ACCS.UR.ZS']
environment135 = sorted(list(wb.topic.members(6)) + others)
len(environment135)

In [ ]:
# Includes TWN
countries = sorted(list(wb.region.members('WLD')))
countries.remove("TWN")
len(countries)

In [ ]:
file_excel = "countries217.csv"
wikipedia = pd.read_csv(os.path.join(out_dir,file_excel))
wikipedia = wikipedia.rename({'Code': 'economy'}, axis=1)
wikipedia.head()

In [ ]:
appended_data = []
for count, variable in enumerate(environment135, start=1):
    print(count, variable)
    df = wb.data.DataFrame(series = variable, 
                    economy = countries, 
                    time = range(2000, 2021, 1), 
                    db=2, labels=True).reset_index()
    df["variable"] = variable
    print("Done")
    appended_data.append(df)
    
appended_data = pd.concat(appended_data)
# appended_data.to_excel('appended.xlsx')

In [ ]:
tableau = appended_data.groupby('variable').count().reset_index()
tableau[['Cat1','Cat2','Cat3','Cat4','Cat5','Cat6']] = tableau['variable'].str.split(expand=True, pat='.')
tableau = tableau.merge(desc, left_on='variable', right_on='variable', suffixes=('_left', '_right')) 
tableau["row"] = tableau['variable'] +": "+ tableau['description']
tableau.set_index('variable', inplace=True)
tableau

In [ ]:
counts = tableau[['Cat1','Cat2']].value_counts().to_frame('counts')
counts.sort_values(by=['Cat1','Cat2'], ascending=True)

In [ ]:
columns = ['YR2000', 'YR2001', 'YR2002','YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008', 'YR2009',
           'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015', 'YR2016','YR2017', 'YR2018', 'YR2019', 'YR2020']

In [ ]:
pd.options.display.float_format = '{:,.1f}'.format
percentages = tableau[columns]/217*100
data = percentages.values.round(1)

In [ ]:
fig = px.imshow(data,
                labels=dict(x = "Year", y = "Indicator", color = "Available"),
                x = columns,
                y = tableau["row"].tolist(), 
                width=1500, 
                height=3000,
                color_continuous_scale = "portland_r"
               )
fig.update_layout(
    title='WDI Availability of Data in Topic 6: Environment (2000-2021)',
    coloraxis_colorbar=dict(
    thicknessmode="pixels", thickness=70,
    lenmode="pixels", len=400,
    yanchor="top", y=.95,
    ticks="outside", ticksuffix=" %",
    dtick=5)
)
fig.update_xaxes(side="top")
fig.write_html("../WDIinventory.html")

# Notes 

https://stackoverflow.com/questions/64778606/plotly-heatmap-color-legend-i-subplothttps://stackoverflow.com/questions/64778606/plotly-heatmap-color-legend-i-subplot

df = wb.data.DataFrame(series = 'EG.ELC.ACCS.RU.ZS', 
                    economy=countries, 
                    time=range(2000, 2021, 1), 
                    db=2, labels=True).reset_index()

In [ ]:
df = wb.data.DataFrame('EG.ELC.ACCS.ZS', wb.region.members('WLD'), time=range(2000, 2021, 1), labels=True)

In [ ]:
wb.series.metadata.get('EG.ELC.ACCS.ZS', economies=['KEN', 'TZA'])